# PBMC 1k 真实数据降维（GitHub 版本）

PCA → t-SNE → kNN graph → UMAP。所有计算均在本 Notebook 内完成。

In [ ]:
import os
import tempfile
temp_dir = tempfile.gettempdir()
os.environ.setdefault("NUMBA_DISABLE_JIT", "1")
os.environ.setdefault("NUMBA_CACHE_DIR", os.path.join(temp_dir, "pbmc_1k_numba"))
os.environ.setdefault("MPLCONFIGDIR", os.path.join(temp_dir, "pbmc_1k_mpl"))

from pathlib import Path
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

sc.settings.verbosity = 1
sc.set_figure_params(dpi=100, dpi_save=300, facecolor="white", frameon=False)

In [ ]:
# 请将 project root directory 改为你自己的路径。
PROJECT_DIR = Path("path/to/your/project")
input_name = "pbmc_1k_feature_selection.h5ad"

project_dir = PROJECT_DIR.expanduser().resolve()

input_path = project_dir / "results" / input_name
figure_dir = project_dir / "results/dimensionality_reduction_figures_github"
output_path = project_dir / "results/pbmc_1k_dimensionality_reduction_from_notebook.h5ad"
figure_dir.mkdir(parents=True, exist_ok=True)

adata = ad.read_h5ad(input_path)

## PCA

In [ ]:
# 使用 normalized expression，同时保留 layers['counts'] 中的 raw counts。
adata.X = adata.layers["scran_normalization"].copy()

sc.pp.pca(
    adata,
    n_comps=50,
    svd_solver="arpack",
    mask_var="highly_deviant",
    random_state=42,
)

sc.pl.pca(adata, color="total_counts", components=["1,2"], show=False)
plt.savefig(figure_dir / "pca_total_counts.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

## t-SNE

In [ ]:
sc.tl.tsne(adata, use_rep="X_pca", perplexity=30, random_state=42)
sc.pl.tsne(adata, color="total_counts", show=False)
plt.savefig(figure_dir / "tsne_total_counts.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

## Neighbor graph and UMAP

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, random_state=42)
sc.tl.umap(adata, min_dist=0.3, random_state=42)

for key in ["total_counts", "pct_counts_mt", "score", "class", "cell_type"]:
    sc.pl.umap(adata, color=key, show=False)
    plt.savefig(figure_dir / f"umap_{key}.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

## 保存结果

In [ ]:
adata.uns["dimensionality_reduction_provenance"] = {
    "input": str(input_path),
    "working_layer": "scran_normalization",
    "feature_mask": "highly_deviant",
    "n_selected_genes": 4000,
    "pca_n_comps": 50,
    "tsne_use_rep": "X_pca",
    "tsne_perplexity": 30,
    "neighbors_n_neighbors": 15,
    "neighbors_n_pcs": 50,
    "umap_min_dist": 0.3,
    "random_state": 42,
}

adata.write(output_path)